# 07b · Fine-Tuning — Colab (T4)

GPU execution path for notebook 07b. Same logic, staged on local disk, with
mixed precision on CUDA.

## In Drive before running
```
DFU_project/
  data/interim/folds_infection.csv
  outputs/results_infection.json     <- notebook 07's frozen baseline
  images.zip                         <- built by make_images_zip.py
  07b_finetune.ipynb                 <- this cell runs it
```

If you already ran `05_feature_extraction_colab.ipynb`, `images.zip` and the
staged image folders are the same ones this notebook needs — no need to rebuild.

## Resume
Each fold checkpoints to `outputs/finetune_ckpt/fold{k}_pred.npz`. If this
notebook disconnects partway through, upload whatever checkpoints exist back to
Drive, restore them in Cell 4 below, and rerun — completed folds are skipped.


In [1]:
# Cell 1 · GPU check
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'no GPU')
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('cuda ready')

Tesla T4
cuda ready


In [29]:
# Cell 2 · mount Drive
from google.colab import drive; drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive')   # edit to match
assert DRIVE.exists(), f'{DRIVE} not found'
print('found', DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
found /content/drive/MyDrive


In [30]:
# Cell 3 · stage images and CSVs, namespaced by corpus
# Same zip structure as notebook 05: roboflow/ and dfuc/ subfolders, so a
# filename shared between corpora cannot silently overwrite the other.
import shutil, time, zipfile
from pathlib import Path

LOCAL = Path('/content/work'); LOCAL.mkdir(exist_ok=True)
(LOCAL/'data/interim').mkdir(parents=True, exist_ok=True)
(LOCAL/'outputs').mkdir(parents=True, exist_ok=True)

for f in ['data/interim/folds_infection.csv', 'data/notebook2_outputs/results_infection.json']:
    src = DRIVE / f
    if src.exists():
        (LOCAL/f).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, LOCAL/f)
        print('copied', f)
    else:
        print('MISSING (optional if not comparing to baseline):', f)

t0 = time.time()
shutil.copy(DRIVE/'data/images.zip', LOCAL/'images.zip')
with zipfile.ZipFile(LOCAL/'images.zip') as z:
    z.extractall(LOCAL/'images')
n_df = sum(1 for _ in (LOCAL/'images/dfuc').glob('*.jpg'))
print(f'staged {n_df:,} dfuc images in {time.time()-t0:.0f}s')
assert n_df > 0, 'expected images/dfuc/ after extraction'

copied data/interim/folds_infection.csv
copied data/notebook2_outputs/results_infection.json
staged 5,372 dfuc images in 3s


In [32]:
# Cell 4 · rewrite paths, restore any prior checkpoints
import pandas as pd
from pathlib import Path

df_index = {p.name: str(p) for p in (LOCAL/'images/dfuc').glob('*.jpg')}
inf = pd.read_csv(LOCAL/'data/interim/folds_infection.csv')
inf['path'] = inf['path'].map(lambda p: df_index.get(Path(str(p)).name))
miss = int(inf.path.isna().sum())
print(f'unmatched: {miss}')
assert miss == 0, 'rebuild images.zip from the current folds_infection.csv'
inf.to_csv(LOCAL/'data/interim/folds_infection.csv', index=False)

# restore checkpoints from a previous interrupted run, if any
ckpt_src = DRIVE/'outputs/finetune_ckpt'
ckpt_dst = LOCAL/'outputs/finetune_ckpt'; ckpt_dst.mkdir(parents=True, exist_ok=True)
if ckpt_src.exists():
    n = 0
    for f in ckpt_src.glob('fold*_pred.npz'):
        shutil.copy(f, ckpt_dst/f.name); n += 1
    print(f'restored {n} fold checkpoints; those folds will be skipped')
else:
    print('no prior checkpoints; starting from fold 0')

unmatched: 0
no prior checkpoints; starting from fold 0


In [36]:
# Cell 5 · run notebook 07b (streams output live)
import subprocess, sys
from pathlib import Path

%cd /content/work
src = DRIVE / 'data/notebooks/07b_finetune.ipynb'
assert src.exists(), f'{src} not found. Upload 07b_finetune.ipynb to Drive.'
import shutil; shutil.copy(src, 'nb07b.ipynb')

subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'python',
                'nb07b.ipynb', '--output', 'nb07b'], check=True)
code = Path('nb07b.py').read_text().replace('raise SystemExit(1)', 'pass')
Path('nb07b.py').write_text(code)

# stream stdout line by line instead of capturing it silently until exit
proc = subprocess.Popen([sys.executable, '-u', 'nb07b.py'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'nb07b.py failed with exit code {proc.returncode}')

/content/work
  the side-by-side comparison. This notebook will still run.
5,372 images, 4,542 units, 1,417 patients
this notebook trains the SAME architecture on the SAME folds as
notebook 07, with only the backbone unfrozen. Everything else is
identical, so the comparison isolates one variable.
device: cuda
dataset and model builder ready (backbone unfrozen)
fold runner ready (checkpoints per fold)
  fold 0: cached (1055 units)
  fold 1: cached (1076 units)
  fold 2: cached (1087 units)
    fold 3 ep0  val AUROC 0.7904  (45s)
    fold 3 ep1  val AUROC 0.8548  (67s)
    fold 3 ep2  val AUROC 0.8964  (90s)
    fold 3 ep3  val AUROC 0.9148  (113s)
    fold 3 ep4  val AUROC 0.9148  (135s)
    fold 3 ep5  val AUROC 0.9302  (158s)
    fold 3 ep6  val AUROC 0.9334  (182s)
    fold 3 ep7  val AUROC 0.9329  (205s)
  fold 3: saved checkpoint (209s total)
    fold 4 ep0  val AUROC 0.7587  (34s)
    fold 4 ep1  val AUROC 0.7972  (57s)
    fold 4 ep2  val AUROC 0.8286  (80s)
    fold 4 ep3  val A

In [35]:
!nvidia-smi

Wed Aug 12 09:37:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [39]:
# Cell 6 · save results and checkpoints back to Drive
import shutil
from pathlib import Path

dst = DRIVE / 'data/notebook2_outputs'; dst.mkdir(parents=True, exist_ok=True)
res = LOCAL/'outputs/results_infection_finetuned.json'
if res.exists():
    shutil.copy(res, dst/res.name)
    print('saved', res.name)

ck_dst = dst/'finetune_ckpt'; ck_dst.mkdir(exist_ok=True)
n = 0
for f in (LOCAL/'outputs/finetune_ckpt').glob('*'):
    shutil.copy(f, ck_dst/f.name); n += 1
print(f'saved {n} checkpoint files')
print('Colab wipes /content on disconnect. Always run this before closing.')

saved results_infection_finetuned.json
saved 5 checkpoint files
Colab wipes /content on disconnect. Always run this before closing.
